In [ ]:
%load_ext autoreload
%autoreload 2 

## Create Instancce

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from simulation_smile.datatypes import FieldOfView
from simulation_smile.sources import Random3D, Random2D, SimpleRing2d, SimpleCircles2d, SimpleLines
from simulation_smile.illumination import Gaussian2D
from simulation_smile.photophysics import Static, Blinking, Ecl
from simulation_smile.optics import CircularAperturePSF
from simulation_smile.detector import SimpleCMOS,SimpleEMCCD
from simulation_smile.pipeline import SimulationPipeline

# 256x256 像素, 每像素 106.7 nm
fov = FieldOfView(height=64, width=64, pixel_size=160)
fov_nm = (fov.width * fov.pixel_size, fov.height * fov.pixel_size)

# 态顺序: [on, dark, bleached]，速率单位 1/s
#   on -> dark   : 5   (~200 ms 后闪灭)
#   dark -> on   : 1    (~1000 ms 后恢复)
#   on -> bleach : 0.2    (~5 s 后漂白)
K = np.array([
    [0.0, 0, 0.0],   # from on
    [0,  0.0, 0.0],   # from dark
    [0.0,  0.0, 0.0],   # from bleached (吸收态)
])


W, H = fov_nm
spacings = np.arange(100, 301, 50)      # [100,150,200,250,300] 组内间距
width    = 20.0
ys = []                                  # 先从上往下排 (组顶相对偏移)
dy = 0.0
for s in spacings:
    for k in range(3):
        ys.append(dy + k * s)            # 组内 3 条: 0, s, 2s
    dy += 2 * s + 500.0                  # 组间最近线距 500

ys = np.array(ys)
ys += H / 2 - (ys.max() + ys.min()) / 2  # 整体上下居中
lines = np.column_stack([
    np.zeros(15),      # angle = 0 (水平)
    np.full(15, W/2),  # cx = 视场中心
    ys,                # cy 从上到下
    np.full(15, width),
])

brightness = np.array([1.0, 0.1, 0.0])   # 只有 on 态发光
burst_probs = [0.4, 0.25, 0.15, 0.1, 0.05, 0.03, 0.02, 0, 0, 0]  

pipe = SimulationPipeline(
    source= SimpleLines(fov_size_nm = fov_nm, lines=lines, density_per_nm=0.04, brightness=500, size=None, min_spacing_nm=None, angle_units="deg"),#SimpleCircles2d(fov_size_nm=fov_nm, radii=[1000,1200,1400,1600], line_density_per_nm=0.015, brightness=10, size=None),
    illumination=Gaussian2D(center_xy=(fov_nm[0] / 2, fov_nm[1] / 2),
                            sigma=10000.0, peak=1.0),
    photophysics = Blinking(rate_matrix=K, state_brightness=brightness, initial_state=0),#Ecl(burst_probs=burst_probs),
    psf=CircularAperturePSF(na=1.49, wavelength=680.0, zernike={11: 0.07, 6: 0.02},
                            pupil_size=128, oversample=4, focus=0),
    detector=SimpleEMCCD(em_gain = 500.0, sensitivity= 3.37, offset = 486.0, bg_noise_e = 0.01, 
                   read_noise_e = 56.0, bg_inhomogeneous= False),
    fov=fov,
    seed=42, 
)
#SimpleCMOS(gain=33.448, read_noise_e=0.2, offset=1893.68, bg_noise_e=0.01, bg_inhomogeneity_frac=False)

result = pipe.run(n_frames=1000, dt=0.10)

print("图像形状:", result.images.shape)
print("发射体数量:", result.ground_truth.n)
print("z 范围: [%.1f, %.1f] nm" % (result.ground_truth.xyz[:, 2].min(),
                                    result.ground_truth.xyz[:, 2].max()))
#print("size 范围: [%.1f, %.1f] nm" % (result.ground_truth.size.min(),result.ground_truth.size.max()))


In [ ]:
psf=CircularAperturePSF(na=1.49, wavelength=680.0, zernike={11: 0.07, 6: 0.02},
                            pupil_size=128, oversample=4, focus=0)
K0 = psf.psf_at(0.0)
n = K0.shape[0]
zs = [-600, -300, 0, 300, 600]
crop = 128
fig, axes = plt.subplots(1, len(zs), figsize=(3.2 * len(zs), 3.2))
for ax_, z in zip(axes, zs):
    K = psf.psf_at(z)
    Kc = K[n // 2 - crop:n // 2 + crop, n // 2 - crop:n // 2 + crop]
    ax_.imshow(Kc, cmap='inferno')
    ax_.set_title(f'z = {z:+d} nm\n(δz={z-psf.focus:+.0f})')
    ax_.axis('off')
plt.suptitle('Defocus sweep'); plt.tight_layout(); plt.show()

In [ ]:
import os
import json
import tifffile
import numpy as np

images_reshaped = result.images.squeeze(axis=1)
print("转换后的形状:", images_reshaped.shape)

save_dir = r"E:\CodingProj\Data\Simulated_Datasets"
os.makedirs(save_dir, exist_ok=True)

base_name = "simulation_1000f_0p10s_simple_lines_no_blinking_500pps_emccd_160nm"
save_path = os.path.join(save_dir, base_name + ".tif")
meta_path = os.path.join(save_dir, base_name + ".json")

# 1. 保存图像
tifffile.imwrite(save_path, images_reshaped)
print(f"图像已保存至: {save_path}")


# 2. 保存 metadata
def _json_default(o):
    """把 numpy 类型转成原生 Python 类型,供 json 序列化。"""
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    return str(o)  # 兜底:实在转不了就存成字符串,不让程序崩


# 顺手补充图像本身的信息,方便日后核对
meta_to_save = dict(result.metadata)
meta_to_save["image_shape"] = list(images_reshaped.shape)
meta_to_save["image_dtype"] = str(images_reshaped.dtype)
meta_to_save["tif_file"] = os.path.basename(save_path)

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta_to_save, f, indent=2, ensure_ascii=False, default=_json_default)

print(f"Metadata 已保存至: {meta_path}")



## 超分辨 ground truth 渲染（σ≈10nm 高斯）并保存

用 `result.ground_truth`（精确亚像素位置）和 `result.photon_stream`
（每个发射体累计总光子数，作强度权重）渲染一张**理想超分辨图**：

- 每个发射体 → S 倍上采样网格上的一个 **2D 高斯**，σ ≈ 10 nm（≈ 原像素 1/10）；
- **高斯积分 = 该发射体累计发射的总光子数**（暗/早漂白的发射体自然更暗甚至不可见）；
- 没有 PSF、没有相机噪声 —— 比纯 delta 点更接近"真实发射体"观感，
  仍可作为 SMLM/超分辨算法的真值靶子，与 SOFI/RIED 等重建结果比对。

In [ ]:
# 用 ground_truth + photon_stream 渲染"超分辨 ground truth"（σ≈10nm 高斯，按总发光强度）并保存
import os
import numpy as np
import tifffile
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

# ---------- 参数 ----------
S = 16                            # 超分辨上采样因子：每个原像素 -> S×S 个亚像素
sigma_nm = fov.pixel_size / 10.0  # ≈10.7 nm，约 1/10 原像素大小的发射体"尺寸"

px = fov.pixel_size
grid_nm = px / S                  # 每个亚像素的物理尺寸 nm

# 1) 强度权重 = 每个发射体在整个电影期间发射的总光子数
tot = result.photon_stream.counts.sum(axis=(1, 2))            # (N,)

# 2) 发射体在高分辨网格上的亚像素坐标（与 PSF.render 同一约定：列=x/px-0.5, 行=y/px-0.5）
col = result.ground_truth.xyz[:, 0] / px - 0.5                # (N,) 原图列
row = result.ground_truth.xyz[:, 1] / px - 0.5                # (N,) 原图行
gc = (col + 0.5) * S - 0.5        # (N,) 高分辨列中心
gr = (row + 0.5) * S - 0.5        # (N,) 高分辨行中心

# 3) 每个发射体 = 一个 σ≈10nm 的 2D 高斯（高斯积分 = 该发射体总光子数）
H, W = result.images.shape[2], result.images.shape[3]
gt = np.zeros((H * S, W * S), dtype=np.float64)
sigma_sp = sigma_nm / grid_nm               # 亚像素单位的 σ
r = int(np.ceil(3 * sigma_sp))              # 截断半径（±3σ）
norm = tot / (2 * np.pi * sigma_sp ** 2)    # 峰高（使高斯积分 = 总光子数）
for i in range(result.ground_truth.n):
    if tot[i] <= 0:
        continue
    c0 = int(np.floor(gc[i])) - r; c1 = int(np.ceil(gc[i])) + r + 1
    r0 = int(np.floor(gr[i])) - r; r1 = int(np.ceil(gr[i])) + r + 1
    if c1 <= 0 or r1 <= 0 or c0 >= W * S or r0 >= H * S:
        continue
    C0, C1 = max(c0, 0), min(c1, W * S)
    R0, R1 = max(r0, 0), min(r1, H * S)
    X = (np.arange(C0, C1) - gc[i]) / sigma_sp
    Y = (np.arange(R0, R1) - gr[i]) / sigma_sp
    gt[R0:R1, C0:C1] += norm[i] * np.exp(-0.5 * (X[None, :] ** 2 + Y[:, None] ** 2))

# 4) 保存 tif（float32，强度 = 光子数）
gt_path = os.path.join(save_dir, base_name + f"_gt_gauss_sresx{S}.tif")
tifffile.imwrite(gt_path, gt.astype(np.float32))
print(f"超分辨 ground truth 已保存: {gt_path}")
print(f"尺寸: {gt.shape}（原 {H}x{W} ×{S}）  σ={sigma_nm:.1f} nm ≈ 原像素/10")
print(f"非零亚像素: {np.count_nonzero(gt)}   累计总光子: {tot.sum():.0f}")

# 5) 显示：全图 + 中心放大（sqrt 色标，亮暗点都可见）
vmax = float(np.percentile(gt[gt > 0], 99.5)) if np.any(gt > 0) else 1.0
fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
im0 = axes[0].imshow(gt, cmap="inferno", norm=PowerNorm(gamma=0.5, vmin=0, vmax=vmax),
                     interpolation="nearest", aspect="equal")
axes[0].set_title(f"Super-res GT（σ≈{sigma_nm:.0f}nm 高斯，强度=总光子）"); axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

r0, r1 = (H // 2 - 20) * S, (H // 2 + 20) * S
c0, c1 = (W // 2 - 20) * S, (W // 2 + 20) * S
im1 = axes[1].imshow(gt[r0:r1, c0:c1], cmap="inferno",
                     norm=PowerNorm(gamma=0.5, vmin=0, vmax=vmax),
                     interpolation="nearest", aspect="equal")
axes[1].set_title("中心 40×40 像素区域放大"); axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046)
plt.tight_layout(); plt.show()

In [ ]:
# 完整显示像素: interpolation='nearest' 关闭插值/抗锯齿, 每个像素呈现为清晰方块;
# aspect='equal' 保证像素为正方形; figsize+dpi 提高显示分辨率
plt.figure(figsize=(8, 8), dpi=100)
plt.imshow(result.images[9, 0], cmap="gray",
           interpolation='nearest', aspect='equal')
plt.title("Phase 1 demo")
plt.colorbar(label="ADU")
plt.tight_layout()
plt.show()
